In [1]:
import pandas as pd
import numpy as np
import kmapper as km
from sklearn.cluster import DBSCAN
import re
import folium
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ==========================================
# 1. CARGA Y PREPARACIÓN DE DATOS
# ==========================================

df_ciclos = pd.read_csv('ciclos_h1_significativos_20_std.csv')
df_sym = pd.read_csv('matriz_tiempos_ponderada_simetrica.csv', index_col=0)

# ¡SOLUCIÓN!: Forzar que los índices y columnas de la matriz sean texto
df_sym.index = df_sym.index.astype(str)
df_sym.columns = df_sym.columns.astype(str)
nodos_totales = df_sym.index.tolist()

df_denue = pd.read_csv('denue_inegi_62_.csv', encoding='latin1')
df_denue = df_denue.rename(columns={'id': 'ID_Unidad'})
df_denue.set_index('ID_Unidad', inplace=True) 

# ¡SOLUCIÓN!: Forzar que el índice del DENUE también sea texto
df_denue.index = df_denue.index.astype(str)

# Alinear los datos del DENUE 
df_denue = df_denue.reindex(nodos_totales)

# ==========================================
# 2. CREACIÓN DE LA LENTE
# ==========================================

nodos_hoyo_global = set()

for lista_string in df_ciclos['Nodos_Formadores']:
    try:
        nodos_lista = re.findall(r'\w+', str(lista_string))
        nodos_hoyo_global.update(nodos_lista)
    except Exception as e:
        print("Error:", e)

# Filtrar solo los nodos formadores que realmente existan en nuestra matriz
nodos_hoyo_validos = [nodo for nodo in nodos_hoyo_global if nodo in df_sym.columns]

print(f"Se encontraron {len(nodos_hoyo_validos)} nodos que conforman los bordes de los ciclos H1.")

# ¡SOLUCIÓN AL WARNING!: Extraer a numpy directamente sin modificar df_sym
lente_global = df_sym[nodos_hoyo_validos].min(axis=1).to_numpy()

# ==========================================
# 3. EJECUCIÓN DE KEPLER MAPPER
# ==========================================

mapper = km.KeplerMapper(verbose=1)

# Asegúrate de ajustar 'eps' si notas que el grafo queda muy desconectado o muy amontonado
clusterer_personalizado = DBSCAN(metric='precomputed', eps=15.0, min_samples=2)

print(lente_global.shape) 
print(df_sym.shape)

# Verificar los primeros valores de la lente antes de mapear
grafo = mapper.map(
    lens=lente_global,
    X=df_sym.to_numpy(),
    clusterer=clusterer_personalizado,
    cover=km.Cover(n_cubes=15, perc_overlap=0.25),
    precomputed=True  # <-- ¡ESTA ES LA LÍNEA MÁGICA QUE FALTA!
)

# ==========================================
# 4. VISUALIZACIÓN INTERACTIVA
# ==========================================

tooltips = np.array([
    f"<b>ID:</b> {idx}<br>"
    f"<b>Lat:</b> {row['latitud']}<br>"
    f"<b>Lon:</b> {row['longitud']}<br>"
    f"<b>Minutos al Hoyo H1:</b> {lente_global[i]:.1f}"
    for i, (idx, row) in enumerate(df_denue.iterrows())
])

mapper.visualize(
    grafo,
    path_html="mapper_multiples_ciclos_h1.html",
    title="Red de Accesibilidad Hospitalaria - Análisis de Ciclos H1",
    custom_tooltips=tooltips,
    color_values=lente_global,
    color_function_name="Minutos al Vacío de Accesibilidad"
)

print("¡Grafo generado con éxito! Abre 'mapper_multiples_ciclos_h1.html' en tu navegador.")

Se encontraron 1504 nodos que conforman los bordes de los ciclos H1.
KeplerMapper(verbose=1)
(3851,)
(3851, 3851)
Mapping on data shaped (3851, 3851) using lens shaped (3851,)

Creating 15 hypercubes.

Created 34 edges and 39 nodes in 0:00:00.339682.
Wrote visualization to: mapper_multiples_ciclos_h1.html
¡Grafo generado con éxito! Abre 'mapper_multiples_ciclos_h1.html' en tu navegador.


In [2]:
'1118' in df_sym.columns.values.tolist()

False

In [3]:
# ==========================================
# 5. PROYECCIÓN GEOGRÁFICA (COLORES SINCRONIZADOS)
# ==========================================

print("Generando mapa geográfico con colores sincronizados con Mapper...")

# 1. Extraer los clústeres para saber quién es ruido y quién no
nodos_mapper = grafo['nodes']

# 2. Configurar EXACTAMENTE la misma paleta que KeplerMapper usa por defecto
# Usamos 'viridis' y normalizamos basándonos en el mínimo y máximo de tus tiempos
cmap = plt.get_cmap('viridis')
norm = mcolors.Normalize(vmin=lente_global.min(), vmax=lente_global.max())

# 3. Crear el mapa base
centro_lat = df_denue['latitud'].mean()
centro_lon = df_denue['longitud'].mean()
mapa_geo = folium.Map(
    location=[centro_lat, centro_lon], 
    zoom_start=11, 
    tiles='CartoDB positron'
)

# 4. Pintar los puntos iterando sobre los datos
for i, (id_unidad, row) in enumerate(df_denue.iterrows()):
    lat = row['latitud']
    lon = row['longitud']
    valor_lente = lente_global[i]
    
    # Buscar en qué clústeres de Mapper aparece
    pertenencias = []
    for cluster_id, indices_nodos in nodos_mapper.items():
        if i in indices_nodos:
            pertenencias.append(cluster_id)
            
    # Asignar color sincronizado
    if len(pertenencias) > 0:
        # Extraemos el color exacto del gradiente viridis según sus minutos de distancia
        # y lo convertimos a formato HEX para que Folium lo entienda
        color_punto = mcolors.to_hex(cmap(norm(valor_lente)))
        texto_clusters = ", ".join(pertenencias)
        opacidad = 0.85
    else:
        # Puntos que DBSCAN descartó como ruido (no están en el grafo de Mapper)
        color_punto = '#cccccc'  # Gris claro
        texto_clusters = "Ninguno (Ruido / Aislado)"
        opacidad = 0.4
        
    html_tooltip = (
        f"<b>ID:</b> {id_unidad}<br>"
        f"<b>Minutos al Hoyo H1:</b> {valor_lente:.1f}<br>"
        f"<b>Clústeres Mapper:</b> {texto_clusters}"
    )
    
    folium.CircleMarker(
        location=[lat, lon],
        radius=6,
        color='#333333', # Borde gris oscuro para definir el punto
        weight=0.5,
        fill=True,
        fill_color=color_punto,
        fill_opacity=opacidad,
        tooltip=html_tooltip
    ).add_to(mapa_geo)

# 5. Guardar el mapa
archivo_mapa = "mapa_geo_sincronizado.html"
mapa_geo.save(archivo_mapa)

print(f"¡Mapa sincronizado generado con éxito! Abre '{archivo_mapa}' en tu navegador.")

Generando mapa geográfico con colores sincronizados con Mapper...
¡Mapa sincronizado generado con éxito! Abre 'mapa_geo_sincronizado.html' en tu navegador.


In [1]:
import pandas as pd
import numpy as np
import kmapper as km
from sklearn.cluster import DBSCAN
import re
import folium
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ==========================================
# 1. CARGA Y PREPARACIÓN DE DATOS
# ==========================================

# ¡IMPORTANTE!: Reemplaza con los nombres exactos de tus archivos
archivo_1_std = "ciclos_h1_significativos_10_std.csv"  # Conjunto A
archivo_2_std = 'ciclos_h1_significativos_20_std.csv'  # Conjunto B

df_ciclos_A = pd.read_csv(archivo_1_std)
df_ciclos_B = pd.read_csv(archivo_2_std)
df_sym = pd.read_csv('matriz_tiempos_ponderada_simetrica.csv', index_col=0)

# Forzar que los índices y columnas sean texto
df_sym.index = df_sym.index.astype(str)
df_sym.columns = df_sym.columns.astype(str)
nodos_totales = df_sym.index.tolist()

df_denue = pd.read_csv('denue_inegi_62_.csv', encoding='latin1')
df_denue = df_denue.rename(columns={'id': 'ID_Unidad'})
df_denue.set_index('ID_Unidad', inplace=True) 
df_denue.index = df_denue.index.astype(str)

# Alinear datos
df_denue = df_denue.reindex(nodos_totales)

# ==========================================
# 2. OPERACIÓN DE CONJUNTOS (A - B) Y LENTE
# ==========================================

def extraer_nodos(df_ciclos):
    nodos = set()
    for lista_string in df_ciclos['Nodos_Formadores']:
        try:
            nodos_lista = re.findall(r'\w+', str(lista_string))
            nodos.update(nodos_lista)
        except Exception as e:
            pass
    return nodos

# Extraer los conjuntos
nodos_A = extraer_nodos(df_ciclos_A) # 1 STD
nodos_B = extraer_nodos(df_ciclos_B) # 2 STD

# Calcular A - B (Nodos que están en 1 STD pero NO en 2 STD)
nodos_A_menos_B = nodos_A - nodos_B

# Filtrar validando contra la matriz
nodos_hoyo_validos = [nodo for nodo in nodos_A_menos_B if nodo in df_sym.columns]

print(f"Nodos en 1 STD: {len(nodos_A)}")
print(f"Nodos en 2 STD: {len(nodos_B)}")
print(f"Nodos exclusivos de 1 STD (A - B) válidos en matriz: {len(nodos_hoyo_validos)}")

# Calcular la lente SOLO respecto a los hoyos pequeños (A - B)
lente_global = df_sym[nodos_hoyo_validos].min(axis=1).to_numpy()

# ==========================================
# 3. EJECUCIÓN DE KEPLER MAPPER
# ==========================================

mapper = km.KeplerMapper(verbose=1)
clusterer_personalizado = DBSCAN(metric='precomputed', eps=15.0, min_samples=2)

grafo = mapper.map(
    lens=lente_global,
    X=df_sym.to_numpy(),
    clusterer=clusterer_personalizado,
    cover=km.Cover(n_cubes=15, perc_overlap=0.25),
    precomputed=True 
)

# Visualización HTML de Mapper
tooltips = np.array([
    f"<b>ID:</b> {idx}<br>"
    f"<b>Lat:</b> {row['latitud']}<br>"
    f"<b>Lon:</b> {row['longitud']}<br>"
    f"<b>Minutos al Hoyo Menor (A-B):</b> {lente_global[i]:.1f}"
    for i, (idx, row) in enumerate(df_denue.iterrows())
])

archivo_mapper = "mapper_ciclos_A_menos_B.html"
mapper.visualize(
    grafo,
    path_html=archivo_mapper,
    title="Red de Accesibilidad - Ciclos Locales (A - B)",
    custom_tooltips=tooltips,
    color_values=lente_global,
    color_function_name="Minutos al Vacío Local"
)
print(f"¡Grafo generado con éxito! Abre '{archivo_mapper}'.")

# ==========================================
# 4. PROYECCIÓN GEOGRÁFICA (FOLIUM) SINCRONIZADA
# ==========================================

print("Generando mapa geográfico sincronizado...")

nodos_mapper = grafo['nodes']
cmap = plt.get_cmap('viridis')
norm = mcolors.Normalize(vmin=lente_global.min(), vmax=lente_global.max())

centro_lat = df_denue['latitud'].mean()
centro_lon = df_denue['longitud'].mean()
mapa_geo = folium.Map(
    location=[centro_lat, centro_lon], 
    zoom_start=11, 
    tiles='CartoDB positron'
)

for i, (id_unidad, row) in enumerate(df_denue.iterrows()):
    lat = row['latitud']
    lon = row['longitud']
    valor_lente = lente_global[i]
    
    pertenencias = []
    for cluster_id, indices_nodos in nodos_mapper.items():
        if i in indices_nodos:
            pertenencias.append(cluster_id)
            
    if len(pertenencias) > 0:
        color_punto = mcolors.to_hex(cmap(norm(valor_lente)))
        texto_clusters = ", ".join(pertenencias)
        opacidad = 0.85
    else:
        color_punto = '#cccccc'
        texto_clusters = "Ninguno (Ruido / Aislado)"
        opacidad = 0.4
        
    html_tooltip = (
        f"<b>ID:</b> {id_unidad}<br>"
        f"<b>Minutos al Hoyo Menor (A-B):</b> {valor_lente:.1f}<br>"
        f"<b>Clústeres Mapper:</b> {texto_clusters}"
    )
    
    folium.CircleMarker(
        location=[lat, lon],
        radius=6,
        color='#333333',
        weight=0.5,
        fill=True,
        fill_color=color_punto,
        fill_opacity=opacidad,
        tooltip=html_tooltip
    ).add_to(mapa_geo)

archivo_mapa = "mapa_geo_sincronizado_A_menos_B.html"
mapa_geo.save(archivo_mapa)
print(f"¡Mapa sincronizado generado con éxito! Abre '{archivo_mapa}'.")

Nodos en 1 STD: 1925
Nodos en 2 STD: 1504
Nodos exclusivos de 1 STD (A - B) válidos en matriz: 421
KeplerMapper(verbose=1)
Mapping on data shaped (3851, 3851) using lens shaped (3851,)

Creating 15 hypercubes.

Created 33 edges and 34 nodes in 0:00:00.421381.
Wrote visualization to: mapper_ciclos_A_menos_B.html
¡Grafo generado con éxito! Abre 'mapper_ciclos_A_menos_B.html'.
Generando mapa geográfico sincronizado...
¡Mapa sincronizado generado con éxito! Abre 'mapa_geo_sincronizado_A_menos_B.html'.
